### Init

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from numpy.fft import rfft, irfft
from matplotlib.animation import FuncAnimation, PillowWriter
import fhd
import importlib
import time
import math
from pathlib import Path
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib as mpl
import pyogrio
from scipy.ndimage import distance_transform_edt
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata
from scipy.interpolate import RBFInterpolator
from scipy.optimize import lsq_linear
from shapely.geometry import box

importlib.reload(fhd)



## Loading files

In [ ]:
DATA_DIR = Path("")

gdf_2011 = 

gdf_2012 = 

gdf_2013 =

gdf_2014 =

gdf_2015 =

gdf_2016 =

gdf_2017 =

gdf_2018 =

gdf_2019 =

gdf_2020 =

# cbsgebiedsindelingen2020.gpkg
area_path = DATA_DIR / "cbsgebiedsindelingen2020.gpkg"

layers = pyogrio.list_layers(area_path)

In [ ]:
gemeenten = gpd.read_file(
    area_path,
    layer="gemeente_niet_gegeneraliseerd"
)

corop = gpd.read_file(
    area_path,
    layer= "coropgebied_gegeneraliseerd"
)

# Replace invalid values with NaN
gdf_2020_clean = gdf_2020.copy()
num_cols = gdf_2020_clean.select_dtypes(include="number").columns
gdf_2020_clean[num_cols] = gdf_2020_clean[num_cols].mask(gdf_2020[num_cols] < -90000, np.nan)

gdf_2019_clean = gdf_2019.copy()
num_cols = gdf_2019_clean.select_dtypes(include="number").columns
gdf_2019_clean[num_cols] = gdf_2019_clean[num_cols].mask(gdf_2019[num_cols] < -90000, np.nan)

gdf_2018_clean = gdf_2018.copy()
num_cols = gdf_2018_clean.select_dtypes(include="number").columns
gdf_2018_clean[num_cols] = gdf_2018_clean[num_cols].mask(gdf_2018[num_cols] < -90000, np.nan)

gdf_2017_clean = gdf_2017.copy()
num_cols = gdf_2017_clean.select_dtypes(include="number").columns
gdf_2017_clean[num_cols] = gdf_2017_clean[num_cols].mask(gdf_2017[num_cols] < -90000, np.nan)

gdf_2016_clean = gdf_2016.copy()
num_cols = gdf_2016_clean.select_dtypes(include="number").columns
gdf_2016_clean[num_cols] = gdf_2016_clean[num_cols].mask(gdf_2016[num_cols] < -90000, np.nan)

gdf_2015_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2014_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2013_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2012_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2011_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

## Functions

In [2]:
def crop_and_rescale(
    year,
    gdf,
    corop_gdf,
    corop_names,
    max_gdf,
    cols,
    cell_size = 500,
    padding_factor=0,
    gemeenten = gemeenten
):
    # --- 0. Determine index based on year ---
    # Change/add if needed
    if year >= 2015 and year < 2017:
        index = 1
    if year >= 2017:  
        index = 0

    # --- 1. Get boundary ---
    boundary = corop_gdf[corop_gdf["statnaam"].isin(corop_names)]
    minx, miny, maxx, maxy = boundary.total_bounds

    width  = maxx - minx
    height = maxy - miny
    side = max(width, height)

    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2
    half = side / 2

    square_bounds = (
        cx - half/2,
        cy - half/2,
        cx + half,
        cy + half,
    )

    # --- 2. Optional padding ---
    padding = padding_factor * cell_size
    square_bounds = (
        square_bounds[0] - padding,
        square_bounds[1] - padding,
        square_bounds[2] + padding,
        square_bounds[3] + padding,
    )

    # --- 3. Crop ---
    clipped = gdf.cx[
        square_bounds[0]:square_bounds[2],
        square_bounds[1]:square_bounds[3]
    ].copy()

    max_clipped = max_gdf.cx[
        square_bounds[0]:square_bounds[2],
        square_bounds[1]:square_bounds[3]
    ]

    gemeenten_clipped = gemeenten.cx[
        square_bounds[0]:square_bounds[2],  
        square_bounds[1]:square_bounds[3]
    ]

    
    # --- 4. Add middle income if needed ---
    if (
        cols[2][index] not in clipped.columns
        and cols[1][index] in clipped.columns
        and cols[3][index] in clipped.columns
    ):
        clipped[cols[2][index]] = (
            100
            - clipped[cols[1][index]]
            - clipped[cols[3][index]]
        )

    # --- 5. Reduce columns ---
    keep_cols = [col[index] for col in cols if col[index] in clipped.columns]
    clipped = clipped[keep_cols + ["geometry"]].copy()

    # --- 6. Rescale ---
    clipped["percentage_leeg"] = (clipped[cols[5][index]] / clipped[cols[4][index]]) 
    
    clipped["percentage_vol"] = 1 - clipped["percentage_leeg"]
    

    for col in [
        cols[index][1],
        cols[index][2],
        cols[index][3],
    ]:
        if col in clipped.columns:
            clipped[f"{col}_rs"] = (
                clipped[col] * clipped["percentage_vol"]
            ) / 100

    return clipped, square_bounds, gemeenten_clipped

NameError: name 'gemeenten' is not defined

In [3]:
def mirrorred(input):
    layers = input.copy()
    nan_mask = np.isnan(layers)   # shape: (3, ny, nx)

    for i in range(3):
        band = layers[i]
        mask = np.isnan(band)

        # indices of nearest non-NaN cell
        _, indices = distance_transform_edt(
            mask,
            return_indices=True
        )
        

        layers[i][mask] = band[tuple(indices[:, mask])]
    return layers, nan_mask

In [4]:
# Needed to build A

def grad_x(f, dx):
    return (np.roll(f, -1, axis=0) - np.roll(f, 1, axis=0)) / (2*dx)

def grad_y(f, dy):
    return (np.roll(f, -1, axis=1) - np.roll(f, 1, axis=1)) / (2*dy)

def laplacian(f, dx, dy):
    return (
        (np.roll(f, -1, axis=0) - 2*f + np.roll(f, 1, axis=0)) / dx**2 +
        (np.roll(f, -1, axis=1) - 2*f + np.roll(f, 1, axis=1)) / dy**2
    )

def divergence(fx, fy, dx, dy):
    return grad_x(fx, dx) + grad_y(fy, dy)

def biharmonic(f, dx, dy):
    return laplacian(laplacian(f, dx, dy), dx, dy)

In [5]:
def build_A(rho, a, dx, dy, beta=1, include_nu=True, include_Gamma=True, scaling=False):
    # build the feature matrix A for species a
    ns, Nx, Ny = rho.shape
    rho0 = 1 - np.sum(rho, axis=0)
    assert ns == 3

    ra = rho[a]
    A = []

    # indices of the other species
    others = [i for i in range(ns) if i != a]
    b, c = others

    rb = rho[b]
    rc = rho[c]

    grad_ra = (grad_x(ra, dx), grad_y(ra, dy))
    grad_rb = (grad_x(rb, dx), grad_y(rb, dy))
    grad_rc = (grad_x(rc, dx), grad_y(rc, dy))
    grad_r0 = (grad_x(rho0, dx), grad_y(rho0, dy))

    rhoa0 = ra * rho0

    # --- 1. Diffusion ---
    fx = rho0 * grad_ra[0] - ra * grad_r0[0]
    fy = rho0 * grad_ra[1] - ra * grad_r0[1]
    A.append(divergence(fx, fy, dx, dy))

    # --- 2–4. Kappa ---
    for grad_r in [grad_ra, grad_rb, grad_rc]:
        fx = rhoa0 * grad_r[0]
        fy = rhoa0 * grad_r[1]
        A.append(-beta * divergence(fx, fy, dx, dy))  # minus from PDE

    # --- 5–10. Nu (reduced set) ---
    if include_nu:
        # ν^{aaa}
        fx = rhoa0 * (ra * grad_ra[0] + ra * grad_ra[0])
        fy = rhoa0 * (ra * grad_ra[1] + ra * grad_ra[1])
        A.append(-beta * divergence(fx, fy, dx, dy))

        # ν^{abb}
        fx = rhoa0 * (rb * grad_rb[0] + rb * grad_rb[0])
        fy = rhoa0 * (rb * grad_rb[1] + rb * grad_rb[1])
        A.append(-beta * divergence(fx, fy, dx, dy))

        # ν^{acc}
        fx = rhoa0 * (rc * grad_rc[0] + rc * grad_rc[0])
        fy = rhoa0 * (rc * grad_rc[1] + rc * grad_rc[1])
        A.append(-beta * divergence(fx, fy, dx, dy))

        # ν^a_ab = ν^{aab} + ν^{aba}
        fx = rhoa0 * (ra * grad_rb[0] + rb * grad_ra[0])
        fy = rhoa0 * (ra * grad_rb[1] + rb * grad_ra[1])
        A.append(-beta * divergence(fx, fy, dx, dy))

        # ν^a_ac
        fx = rhoa0 * (ra * grad_rc[0] + rc * grad_ra[0])
        fy = rhoa0 * (ra * grad_rc[1] + rc * grad_ra[1])
        A.append(-beta * divergence(fx, fy, dx, dy))

        # ν^a_bc
        fx = rhoa0 * (rb * grad_rc[0] + rc * grad_rb[0])
        fy = rhoa0 * (rb * grad_rc[1] + rc * grad_rb[1])
        A.append(-beta * divergence(fx, fy, dx, dy))
    else:
        A.append(np.zeros_like(ra))  # placeholder for ν^{aaa}
        A.append(np.zeros_like(ra))  # placeholder for ν^{abb}
        A.append(np.zeros_like(ra))  # placeholder for ν^{acc}
        A.append(np.zeros_like(ra))  # placeholder for ν^a_ab
        A.append(np.zeros_like(ra))  # placeholder for ν^a_ac
        A.append(np.zeros_like(ra))  # placeholder for ν^a_bc

    if include_Gamma:
        # --- 11. Gamma ---
        lap_ra = laplacian(ra, dx, dy)
        grad_lap_ra = (grad_x(lap_ra, dx), grad_y(lap_ra, dy))

        fx = rhoa0 * grad_lap_ra[0]
        fy = rhoa0 * grad_lap_ra[1]
        A.append(-beta * divergence(fx, fy, dx, dy))
    else:
        A.append(np.zeros_like(ra))  # placeholder for Gamma

    # --- stack into matrix ---
    A = np.stack([f.reshape(-1) for f in A], axis=1)

    if scaling:
        # --- normalize ---
        scales = np.std(A, axis=0) + 1e-12
        A /= scales
        return A, scales

    return A, None

In [6]:
def unpack_theta(Thetas, ns=3):
    """
    Convert list of theta_a (length 11 each) into:
        D:     (ns,)
        kappa: (ns, ns)
        nu:    (ns, ns, ns)   # nu[a, b, c]
        Gamma: (ns, ns)       # each row a filled with Gamma^a (repeated)
    """
    assert ns == 3, "This implementation assumes ns=3"
    assert len(Thetas) == ns

    D = np.zeros(ns)
    kappa = np.zeros((ns, ns))
    nu = np.zeros((ns, ns, ns))
    Gamma = np.zeros((ns, ns))

    for a in range(ns):
        theta = Thetas[a]
        assert len(theta) == 11

        # indices of the other species
        others = [i for i in range(ns) if i != a]
        b, c = others

        # --- 1. D ---
        D[a] = theta[0]

        # --- 2. kappa ---
        # order: κ_aa, κ_ab, κ_ac
        kappa[a, a] = theta[1]
        kappa[a, b] = theta[2]
        kappa[a, c] = theta[3]

        # --- 3. nu (fill symmetric in b,c) ---
        # order:
        # ν_aaa, ν_abb, ν_acc, ν^a_ab, ν^a_ac, ν^a_bc

        # diagonal blocks
        nu[a, a, a] = theta[4]
        nu[a, b, b] = theta[5]
        nu[a, c, c] = theta[6]

        # mixed terms (split symmetric pairs)
        nu[a, a, b] = nu[a, b, a] = theta[7] / 2
        nu[a, a, c] = nu[a, c, a] = theta[8] / 2
        nu[a, b, c] = nu[a, c, b] = theta[9] / 2

        # --- 4. Gamma ---
        # repeat Gamma^a across row a (your requested format)
        Gamma[a, :] = theta[10]

    return D, kappa, nu, Gamma

In [ ]:
def compute_thetas(mirror_start, mirror_end, nan_mask_start, dx, dy, ns, delta_t, include_nu=True, include_Gamma=True, scaling=True):
    # Compute time derivative
    partial_m = np.zeros_like(mirror_start)
    for i in range(mirror_start.shape[0]):
        partial_m[i] = (mirror_end[i] - mirror_start[i]) / delta_t

    thetas = []
    stds = []

    for a in range(ns):
        drho_dt_a = partial_m[a].reshape(-1)

        A, scales = build_A(
            mirror_start, a, dx, dy,
            include_nu=True, include_Gamma=True, scaling=True
        )

        nan_mask_flat = nan_mask_start[a].reshape(-1)
        drho_dt_a = drho_dt_a[~nan_mask_flat]
        A = A[~nan_mask_flat]

        stds.append(np.std(A, axis=0))

        # Bounds: first parameter >= 0, others free
        n_params = A.shape[1]
        lb = np.full(n_params, -np.inf)
        ub = np.full(n_params,  np.inf)
        lb[0] = 0.0

        res = lsq_linear(A, drho_dt_a, bounds=(lb, ub))
        theta_a = res.x

        if scales is not None:
            theta_a = theta_a * scales

        thetas.append(theta_a)

    return thetas, stds

## Crop to Area (The Hague-Rotterdam)

In [ ]:
# Selected areas
corop_names = [
    "Agglomeratie 's-Gravenhage",
    "Delft en Westland",
    "Groot-Rijnmond",
]

# First column = column-name in >2017 CBS data
# Second column = column_name in <2017 CBS data
# Third column = column_name in 2011 CBS data
# etc.
cols = [ 
    ("aantal_inwoners", "INWONER","INW2011", "INW2012", "INW2013", "INW2014", "Aantal inwoners"),
    ("percentage_laag_inkomen_huishouden", "P_LINK_HH","P_LINH2011", "P_LINH2012", "P_LINH2013", "P_LINH2014", "Laag inkomen"),
    ("percentage_midden_inkomen_huishouden", "P_MINK_HH","P_MIDDEL2011", "P_MIDDEL2012", "P_MIDDEL2013", "P_MIDDEL2014", "Midden inkomen"),
    ("percentage_hoog_inkomen_huishouden", "P_HINK_HH","P_HINH2011", "P_HINH2012", "P_HINH2013", "P_HINH2014", "Hoog inkomen"),
    ("aantal_woningen", "WONING"),
    ("aantal_niet_bewoonde_woningen", "WON_NBEW"),
    ("gemiddelde_woz_waarde_woning")
]

# Repeat for every year
dhr_gdf_reduced_2015, bounds_dhr, gemeenten_clipped_2015 = crop_and_rescale(
    year=2015,
    gdf=gdf_2015_clean,
    corop_gdf=corop,
    corop_names=corop_names,
    cell_size=500,
    max_gdf=inw_gdf_clean,
    cols=cols
)

## Fill grid

In [ ]:
cell_size = 500 #Change to 100
n = int(bounds_dhr[2]-bounds_dhr[0]) // cell_size

ns = 3
N = (n, n)
Lx, Ly = (50,50)
dx = Lx/n
dy = Ly/n

x = np.arange(-Lx/2, Lx/2, dx)
y = np.arange(-Ly/2, Ly/2, dy)

minx, miny, maxx, maxy = bounds_dhr

centroids = dhr_gdf_reduced_2015.geometry.centroid

cols = ((centroids.x - minx) // cell_size).astype(int)
rows = ((centroids.y - miny) // cell_size).astype(int)

cols = np.clip(cols, 0, len(x)-1)
rows = np.clip(rows, 0, len(y)-1)

cols_inc = [ 
    ("percentage_laag_inkomen_huishouden", "P_LINK_HH","P_LINH2011", "P_LINH2012", "P_LINH2013", "P_LINH2014", "Laag inkomen"),
    ("percentage_midden_inkomen_huishouden", "P_MINK_HH","P_MIDDEL2011", "P_MIDDEL2012", "P_MIDDEL2013", "P_MIDDEL2014", "Midden inkomen"),
    ("percentage_hoog_inkomen_huishouden", "P_HINK_HH","P_HINH2011", "P_HINH2012", "P_HINH2013", "P_HINH2014", "Hoog inkomen"),
]

inx = 1 # Change for year to right column index
# Repeat for every year

# 2015
phi_rdh_2015 = np.full((ns, len(y), len(x)), np.nan)

# Fraction of each species
for i in range(ns):
    col_name = cols_inc[i][inx]
    phi_rdh_2015[i, rows, cols] = dhr_gdf_reduced_2015[col_name].fillna(np.nan) / 100

# Total resedents (inwoners)
inw_rhd_2015[rows, cols] = dhr_gdf_reduced_2015[cols_inw[inx]].fillna(np.nan)


In [ ]:
# Repeat for each year

mirror_2015, nan_mask_2015 = mirrorred(phi_rdh_2015)
mirror_2016, nan_mask_2016 = mirrorred(phi_rdh_2016)
mirror_2017, nan_mask_2017 = mirrorred(phi_rdh_2017)
mirror_2018, nan_mask_2018 = mirrorred(phi_rdh_2018)
mirror_2019, nan_mask_2019 = mirrorred(phi_rdh_2019)
mirror_2020, nan_mask_2020 = mirrorred(phi_rdh_2020)

# Inference

### 2015-2016 (repeat for each year)

In [ ]:
D_t = 1

Thetas_16_15, std_16_15 = compute_thetas(
    mirror_2015,
    mirror_2016,
    nan_mask_2015, # Mask from earliest year
    dx, dy,
    ns,
    delta_t = D_t
)

D_fit, kappa_fit, nu_fit, Gamma_fit = unpack_theta(Thetas_16_15, ns=3)
Gamma_fit *= np.eye(ns)

D_fit = D_fit / D_t
kappa_fit = kappa_fit / D_t
nu_fit = nu_fit / D_t
Gamma_fit = Gamma_fit / D_t

param_fitted_16_15 = {
    'D': D_fit,
    'kappa': kappa_fit,
    'Gamma': Gamma_fit,
    'nu': nu_fit,
}

### Simulation for performance

In [ ]:
N = (n-1, n-1)
L = (Lx, Ly)
sim_2d_fitted = fhd.fhd_2d_3species(L,N, bc= 'Neumann', fft=False)

time_step = 1
dt = 1
nsteps = int(time_step / dt)
noise = 0
frames = 1

st = time.time()
phi_run_fitted_2015 = sim_2d_fitted.run(mirror_2015, param_fitted_16_15, nsteps, dt, noise, frames, model = "Vitelli", verbatum = False)
et = time.time()
print(f"Simulation ran in t = {et-st:.6f} seconds")

phi_masked_2015 = phi_run_fitted_2015.copy()
phi_masked_2015[:,-1][nan_mask_2016] = 0.0 # Mask with year of end simulation 

### Figure (optional)

In [ ]:
sim_2d_fitted.phi_plot(phi_masked_2015[:,-1], param_fitted_15, model = "Vitelli", plot_2 = False, plot_3 = False)

### Save Theta

In [ ]:
# For the basic model
thetas_basic = { '2015-2016': Thetas_16_15, '2016-2017' : Thetas_17_16, '2017-2018' : Thetas_18_17, '2018-2019' : Thetas_19_18, '2019-2020' : Thetas_20_19}

# Parameter plot

In [ ]:
params_title = ['D', 'kappa_a', 'kappa_b', 'kappa_c', 'nu_aa', 'nu_bb', 'nu_cc', 'nu_ab', 'nu_ac', 'nu_bc', 'Gamma']

params_a_name = ['Ds_a', 'kappa_aas_a', 'kappa_abs_a', 'kappa_acs_a', 'nu_aas_a', 'nu_bbs_a', 'nu_ccs_a', 'nu_abs_a', 'nu_acs_a', 'nu_bcs_a', 'Gammas_a']
params_b_name = ['Ds_b', 'kappa_aas_b', 'kappa_abs_b', 'kappa_acs_b', 'nu_aas_b', 'nu_bbs_b', 'nu_ccs_b', 'nu_abs_b', 'nu_acs_b', 'nu_bcs_b', 'Gammas_b']
params_c_name = ['Ds_c', 'kappa_aas_c', 'kappa_abs_c', 'kappa_acs_c', 'nu_aas_c', 'nu_bbs_c', 'nu_ccs_c', 'nu_abs_c', 'nu_acs_c', 'nu_bcs_c', 'Gammas_c']
years = ['2015', '2016', '2017', '2018', '2019']

Ds_a = [thetas_basic[year][0][0] for year in years]
kappa_aas_a = [thetas_basic[year][0][1] for year in years]
kappa_abs_a = [thetas_basic[year][0][2] for year in years]
kappa_acs_a = [thetas_basic[year][0][3] for year in years]
nu_aas_a = [thetas_basic[year][0][4] for year in years]
nu_bbs_a = [thetas_basic[year][0][5] for year in years]
nu_ccs_a = [thetas_basic[year][0][6] for year in years]
nu_abs_a = [thetas_basic[year][0][7] for year in years]
nu_acs_a = [thetas_basic[year][0][8] for year in years]
nu_bcs_a = [thetas_basic[year][0][9] for year in years]
Gammas_a = [thetas_basic[year][0][10] for year in years]

Ds_b = [thetas_basic[year][1][0] for year in years]
kappa_aas_b = [thetas_basic[year][1][1] for year in years]
kappa_abs_b = [thetas_basic[year][1][2] for year in years]
kappa_acs_b = [thetas_basic[year][1][3] for year in years]
nu_aas_b = [thetas_basic[year][1][4] for year in years]
nu_bbs_b = [thetas_basic[year][1][5] for year in years]
nu_ccs_b = [thetas_basic[year][1][6] for year in years]
nu_abs_b = [thetas_basic[year][1][7] for year in years]
nu_acs_b = [thetas_basic[year][1][8] for year in years]
nu_bcs_b = [thetas_basic[year][1][9] for year in years]
Gammas_b = [thetas_basic[year][1][10] for year in years]

Ds_c = [thetas_basic[year][2][0] for year in years]
kappa_aas_c = [thetas_basic[year][2][1] for year in years]
kappa_abs_c = [thetas_basic[year][2][2] for year in years]
kappa_acs_c = [thetas_basic[year][2][3] for year in years]
nu_aas_c = [thetas_basic[year][2][4] for year in years]
nu_bbs_c = [thetas_basic[year][2][5] for year in years]
nu_ccs_c = [thetas_basic[year][2][6] for year in years]
nu_abs_c = [thetas_basic[year][2][7] for year in years]
nu_acs_c = [thetas_basic[year][2][8] for year in years]
nu_bcs_c = [thetas_basic[year][2][9] for year in years]
Gammas_c = [thetas_basic[year][2][10] for year in years]

species = ["Low income", "Middel income", "High income"]
species_colors = ["red", "green", "blue"]


# Performance

In [ ]:
# Fit for 2016 is model run from previous year
# Repeat for each year
model_fit_2016 = phi_masked_2015.copy()[:,-1]

model_fit_2017 = phi_masked_2016.copy()[:,-1]

model_fit_2018 = phi_masked_2017.copy()[:,-1]

model_fit_2019 = phi_masked_2018.copy()[:,-1]

model_fit_2020 = phi_masked_2019.copy()[:,-1]

In [8]:
n_years = 6

bases = np.zeros(n_years-1)
basic = np.zeros(n_years-1)

for i, year in enumerate(range(2015, 2020)): # Change to correct years
    data_year = eval(f"data_{year}")
    data_next_year = eval(f"data_{year+1}")
    bases[i] = np.mean((data_year - data_next_year)**2)

for i, year in enumerate(range(2016, 2021)):
    data_year = eval(f"data_{year}")
    model_year_basic = eval(f"model_fit_{year}")
    basic[i] = np.mean((model_year_basic - data_year)**2) / bases[i]


NameError: name 'np' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
years = [2016, 2017, 2018, 2019, 2020]
base_line = np.ones_like(bases)
plt.plot(years, base_line, marker='o', label='Baseline (data to data)', color = 'red', linestyle = 'dashed')
plt.plot(years, basic, marker='o', label='Basic model')

plt.xlabel('Year', size = axes_size)
plt.ylabel('Normalized MSE', size = axes_size)
plt.title('Normalized MSE of different models over time', size = title_size)
plt.xticks(years)
plt.legend()
plt.show()